# OpenDDE Structure Prediction

![OpenDDE Structure Prediction](https://proto-bio.github.io/proto-assets/images/tool/opendde/hero.png)

This notebook demonstrates all-atom structure prediction using OpenDDE, an open-source co-folding model developed by Aureka AI Research. OpenDDE jointly predicts the three-dimensional structures of proteins, nucleic acids (DNA and RNA), small-molecule ligands, and their complexes using a diffusion-based generative approach with configurable sampling and recycling steps. We demonstrate `run_opendde` by folding the Trp-cage mini-protein and human insulin, covering input construction, model configuration, result inspection, visualization, and export.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("opendde")
display_overview("opendde")
display_docs_section("opendde", "Background")

# OpenDDE

OpenDDE is [Aureka AI Research](https://github.com/aurekaresearch)'s open-source, all-atom biomolecular co-folding foundation model in the AlphaFold3 family: a single model that predicts the joint 3D structure of complexes mixing proteins, DNA, RNA, small-molecule ligands, and ions. This toolkit runs OpenDDE structure prediction on a local GPU, with optional multiple-sequence alignments, and returns per-complex confidence metrics.

OpenDDE ([Aureka AI Research, 2026](https://arxiv.org/abs/2607.03787)) predicts the joint 3D structure of a biomolecular assembly from the sequences and chemical components it contains. It is an openly licensed, all-atom co-folding model where one model folds complexes that mix proteins, DNA, RNA, and small-molecule ligands and predicts how those components are arranged relative to one another. Each protein chain can be paired with a multiple-sequence alignment (MSA) of evolutionarily related sequences, whose covariation patterns supply the evolutionary signal the model uses to place residues.

Architecturally, OpenDDE follows AlphaFold3: it carries a single representation of the input tokens and a pairwise representation over token pairs, refines them through a Pairformer-style trunk, and generates all-atom coordinates with a diffusion module that starts from noise and iteratively denoises into a structure. Several structures can be sampled per complex and ranked by a confidence score. Predicted confidence includes a per-residue predicted local distance difference test (pLDDT) for local reliability, a global predicted distance error (gPDE) for the relative placement of tokens, and predicted template-modeling (pTM) and interface predicted template-modeling (ipTM) scores that summarize overall and interface accuracy, together with an overall ranking score used to select the best sample.

The reference implementation is open-sourced at [aurekaresearch/OpenDDE](https://github.com/aurekaresearch/OpenDDE), with both the code and the model parameters released under the Apache-2.0 license for academic and commercial use. It builds on ideas and components from [Protenix](https://github.com/bytedance/Protenix), [OpenFold](https://github.com/aqlaboratory/openfold), and [ColabFold](https://github.com/sokrypton/ColabFold). Two checkpoints are released: a general-purpose model and an antibody-antigen-tuned variant. It was developed by Aureka AI Research as an open drug-discovery engine spanning structure prediction, design, and optimization.

## Available tools

In [2]:
display_available_tools("opendde")

- **`run_opendde()`** — All-atom biomolecular structure prediction using OpenDDE

### `run_opendde`

OpenDDE predicts 3D structures of proteins, DNA, RNA, ligands, and their complexes using a diffusion-based generative model. It supports both single-sequence mode and MSA-assisted prediction via MMseqs2 homology search. The `num_cycles` parameter controls iterative structural refinement, while `num_steps` governs the granularity of the denoising process. When `num_samples` is set above 1, OpenDDE generates multiple independent structure samples and returns the best by ranking score, which is useful for exploring conformational diversity. Ligands are provided as SMILES strings or CCD codes and are automatically converted to the appropriate internal representation.

In [3]:
from pathlib import Path

from proto_tools import (
    Chain,
    Complex,
    OpenDDEConfig,
    OpenDDEInput,
    run_opendde,
)

In [4]:
# Display input docs
display_api_reference("opendde", "input", "run_opendde")

# Trp-cage TC5b — a 20-residue mini-protein that adopts a compact fold
trpcage_sequence = "NLYIQWLKDGGPSSGRPPPS"

# Create a single-protein complex
complex = Complex(chains=[Chain(sequence=trpcage_sequence, entity_type="protein")])

# Create input
inputs = OpenDDEInput(complexes=[complex])

**Input** — `OpenDDEInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>complexes</code> | <code>list[Complex]</code> | required | List of complexes to predict structure for containing chains and entity types. |
| <code>msas</code> | <code>list[ComplexMSAs] &#124; None</code> | <code>None</code> | Per-complex MSAs; a bare dict[int, MSA] is coerced to an unpaired ComplexMSAs. |

In [5]:
# Display config docs
display_api_reference("opendde", "config", "run_opendde")

# Configure OpenDDE with reduced settings for a fast demonstration run
config = OpenDDEConfig(
    verbose=False,
    device="cuda",  # Change to "cpu" if no GPU available
    use_msa=False,  # single-sequence mode for a fast first prediction
    num_samples=1,
)

**Config** — `OpenDDEConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on (e.g., 'cuda', 'cpu') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>1200</code> | Maximum execution time in seconds. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>include_pae_matrix</code> | <code>bool</code> | <code>False</code> | Attach the full per-residue PAE matrix. |
| <code>use_msa</code> | <code>bool</code> | <code>True</code> | Whether to auto-generate MSAs via MMseqs2 homology search; supplied MSAs are always used |
| <code>msa_search_config</code> | <code>Mmseqs2HomologySearchConfig &#124; None</code> | <code>None</code> | Nested MMseqs2 homology-search config for MSA generation; None uses default settings. |
| <code>pair_heterocomplex_msas</code> | <code>bool</code> | <code>True</code> | Whether heterocomplex protein chains should use taxonomy-paired MSA generation. |
| <code>name</code> | <code>str</code> | <code>'opendde_job'</code> | Name of the OpenDDE folding job; drives the output path. |
| <code>model_checkpoint</code> | <code>str</code> | <code>'opendde_v1'</code> | Bundled model name ('opendde_v1' or 'opendde_abag') or a path to a custom .pt checkpoint. |
| <code>num_samples</code> | <code>int</code> | <code>1</code> | Independent diffusion samples per complex (--sample); best by ranking score is kept. |
| <code>num_steps</code> | <code>int</code> | <code>200</code> | Diffusion denoising steps (--step). Higher = more refined but slower. |
| <code>num_cycles</code> | <code>int</code> | <code>10</code> | Recycling iterations (--cycle). Higher = more accurate but slower. |
| <code>use_template</code> | <code>bool</code> | <code>False</code> | Enable OpenDDE's template search pipeline (--use_template). |
| <code>use_rna_msa</code> | <code>bool</code> | <code>False</code> | Enable OpenDDE's RNA MSA pipeline (--use_rna_msa). |

In [6]:
# Run structure prediction
result = run_opendde(inputs, config)

Folding structures (OpenDDE):   0%|          | 0/1 [00:00<?, ?complex/s]

In [7]:
# Display output docs
display_api_reference("opendde", "output", "run_opendde")

trpcage_structure = result.structures[0]

# Print confidence metrics
print(f"  Number of chains:  {len(complex.chains)}")
print(f"  Protein length:    {len(trpcage_sequence)} residues")
print(f"  Average pLDDT:     {trpcage_structure.metrics.avg_plddt:.1f}")
print(f"  pTM score:         {trpcage_structure.metrics.ptm:.3f}")
print(f"  Ranking score:     {trpcage_structure.metrics.ranking_score:.3f}")

**Output** — `OpenDDEOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>structures</code> | <code>list[Structure]</code> | required | List of predicted structures, one per input complex. |

**Metrics** — `OpenDDEMetrics` (one set per `structures` item)

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>avg_plddt</code> **(primary)** | <code>float</code> | <code>[0, 100]</code> |  |  |
| <code>ptm</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>iptm</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>gpde</code> | <code>float</code> | <code>&gt;= 0</code> |  |  |
| <code>ranking_score</code> | <code>float</code> |  |  |  |
| <code>has_clash</code> | <code>bool</code> |  |  |  |
| <code>avg_pae</code> | <code>float</code> | <code>[0, 32]</code> |  |  |
| <code>pae</code> | <code>list[list[float]]</code> | <code>[0, 32]</code> |  |  |

  Number of chains:  1
  Protein length:    20 residues
  Average pLDDT:     93.8
  pTM score:         0.459
  Ranking score:     0.092


#### Visualize the predicted structure

The interactive viewer renders the predicted mini-protein colored by pLDDT confidence, allowing you to inspect the fold geometry and per-residue confidence.

> NOTE: The 3D viewer below renders locally (JupyterLab, VS Code) but not in GitHub previews.

In [8]:
trpcage_structure.visualize(style="cartoon", color_by="bfactor")

#### Predict a multi-chain complex

OpenDDE can fold multi-chain complexes and report an interface confidence (`iptm`) between chains. Here we fold human insulin, a two-chain complex whose A and B chains associate through disulfide bonds, and tune the sampling configuration: `num_samples=2` keeps the best of two diffusion samples (by ranking score), while `num_steps=100` trades a little accuracy for speed.

In [9]:
# Human insulin — two-chain protein complex (A chain + B chain)
insulin_a_chain = "GIVEQCCTSICSLYQLENYCN"
insulin_b_chain = "FVNQHLCGSHLVEALYLVCGERGFFYTPKT"

insulin_complex = Complex(
    chains=[
        Chain(sequence=insulin_a_chain, entity_type="protein"),
        Chain(sequence=insulin_b_chain, entity_type="protein"),
    ]
)
insulin_inputs = OpenDDEInput(complexes=[insulin_complex])

insulin_config = OpenDDEConfig(
    verbose=False,
    device="cuda",  # Change to "cpu" if no GPU available
    model_checkpoint="opendde_v1",
    num_samples=2,  # keep the best of 2 diffusion samples by ranking score
    num_steps=100,  # fewer denoising steps for a faster demo
    use_msa=False,
)

insulin_result = run_opendde(insulin_inputs, insulin_config)
insulin_structure = insulin_result.structures[0]
print(f"  Chains:         {len(insulin_complex.chains)}")
print(f"  Average pLDDT:  {insulin_structure.metrics.avg_plddt:.1f}")
print(f"  pTM score:      {insulin_structure.metrics.ptm:.3f}")
print(f"  ipTM score:     {insulin_structure.metrics.iptm:.3f}")  # interface confidence between chains

Folding structures (OpenDDE):   0%|          | 0/1 [00:00<?, ?complex/s]

  Chains:         2
  Average pLDDT:  79.5
  pTM score:      0.615
  ipTM score:     0.497


## Export Results

Predicted structures can be exported to PDB or mmCIF format for downstream analysis in molecular visualization tools such as PyMOL, ChimeraX, or VMD. The B-factor column contains pLDDT confidence scores for per-residue visualization.

In [10]:
# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export results to mmCIF (B-factor column carries per-residue pLDDT)
insulin_result.export(name="insulin_complex", export_path=output_dir, file_format="cif")

# Export results to PDB for tools like PyMOL / ChimeraX
insulin_result.export(name="insulin_complex", export_path=output_dir, file_format="pdb")
print(f"Structure exported to {output_dir / 'insulin_complex.cif'}")

Structure exported to example_output/insulin_complex.cif
